## 1-  Explicación Capa Bronze

### Este script realiza una **carga de archivos CSV desde Azure Data Lake Storage**, utilizando **secretos seguros**, y los convierte en tablas Delta Lake organizadas bajo un esquema en capa bronze.

---

### Importacion de modulos
Se importan los modulos especificos, en este caso para evitar cualquier uso no declarado de un modulo futuro usamos ( * )
 
Permite usar SparkSession.builder.getOrCreate() sin importar la clase explícitamente.

In [0]:
# BRONZE - Ingesta de datos crudos
from pyspark.sql import *
from pyspark.sql.functions import *


## 2- Creacion de Databricks secrets scope y key 
Sirve para crear y guardar una clave/token de acceso al almacenamiento en Databricks de forma segura, usando comandos de shell.

Databricks Secrets : Permite ver lista de comandos especificos

Databricks secrets create-scope "scopefinal": Creo un scope que alamcena la futura key o token del ADLS

Databricks secret put-secret "scopefinal" "keyfinal" : Comando para crear y asignar al scope la definida key

AVISO: Esto se ejecuta una sola vez por eso esta comentado, si es la primera vez que ejecuta esta notebook entonces quitar comentarios y usar a su disposición



In [0]:
# %sh
# databricks secret ##COMANDO PARA VER LISTA DE COMANDOS DISPONIBLES
# databricks secret create-scope "scopefinal" ###COMANDO PARA CREAR SCOPE SECRET, es una "varibale" con nombre que luego va a almacenar la key token de su blob storage
# databricks secret put-secret "scopefinal" "keyfinal"

## 3- Recuperamos de forma segura el valor de el secreto almacenado en Databricks.

In [0]:

dbutils.secrets.get(scope="scopefinal", key="keyfinal")

## 4- Recupera desde Databricks un valor secreto (por ejemplo, una URL del tipo_ abfss://..._) y lo guarda en la variable storage_path

## _dbutils.fs.ls(storage_path)_ : Usa esa ruta _(storage_path)_ para listar los archivos que hay en esa ubicación del almacenamiento

In [0]:
storage_path = dbutils.secrets.get(scope="scopefinal", key="keyfinal")

dbutils.fs.ls(storage_path)

## 5- Este siguiente script lista todos los archivos CSV dentro de una carpeta _(storage_path)_, carga cada uno como un DataFrame de Spark, lo guarda en un diccionario dfs y además crea una vista temporal con su nombre (sin extensión) para poder hacer consultas SQL sobre ellos.

In [0]:
# Listar archivos CSV en el directorio usando la ruta del secreto
files = dbutils.fs.ls(storage_path)

# Crear un diccionario para almacenar DataFrames por archivo
dfs = {}

for file in files:
    if file.name.endswith(".csv"):
        path = file.path
        df_temp = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(path)
        file_name_without_extension = file.name[:-4]
        if '.' in file_name_without_extension:
            file_name_without_extension = file_name_without_extension.split('.')[-1]
        dfs[file_name_without_extension] = df_temp
        # Crear vista temporal para cada DataFrame
        df_temp.createOrReplaceTempView(file_name_without_extension)

# Mostrar los DataFrames cargados
for name, df_individual in dfs.items():
    print(f"DataFrame for {name}:")
    display(df_individual)

## 6 - Creamos Schemas en el catalogo especifico (capa_bronze.tables) y luego listamos cada tabla dentro del mismo catalogo mediante iteracion _FOR_

In [0]:
spark.sql(f"""
          CREATE SCHEMA IF NOT EXISTS capa_bronze.tables
          """)

for name in dfs.keys():
    spark.sql(f"""
              CREATE TABLE IF NOT EXISTS capa_bronze.tables.{name} AS
              SELECT * FROM {name}
              """)

In [0]:
spark.sql(
    """
        select * from transactions
    """
).display()

## 7 - Este siguiente script hace una lectura masiva de archivos CSV desde un Data Lake y los guarda como tablas Delta en la capa bronze en la ubicacion respectiva

In [0]:

tables = ["account", "customers","loan_payments","loans","transactions"] 

# Mediante iteracion for leemos y guardamos los csv en delta
for table in tables:
    df_tables = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("sep", ",")
          .option("schema", "string")
          .csv(f"abfss://raw@mistorageprincipal.dfs.core.windows.net/origenPrestamosBanco/dbo.{table}.csv"))
    
    df_tables.write.format("delta").mode("overwrite").save(f"abfss://bronze@mistorageprincipal.dfs.core.windows.net/delta_tables/{table}")

### FINALIZACION CAPA _BRONZE_
---